In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

def preprocess_data(train_df, test_df):
    """
    銀行顧客離脱データセットの前処理および特徴量エンジニアリングを行う関数
    """
    print("前処理を開始します...")
    
    # 1. 正解ラベルの分離とIDの退避
    y_train = train_df['Exited'] if 'Exited' in train_df.columns else None
    
    # 学習用とテスト用を区別するためのフラグを立てて一時的に結合
    train_features = train_df.drop(columns=['Exited']) if 'Exited' in train_df.columns else train_df.copy()
    test_features = test_df.copy()
    
    train_features['is_train'] = 1
    test_features['is_train'] = 0
    
    combined = pd.concat([train_features, test_features], axis=0).reset_index(drop=True)
    
    # 2. 名字（Surname）の数（Frequency Encoding）
    # 同じ苗字がデータ中に何回登場するか（家族ルールの抽出）
    surname_counts = combined['Surname'].value_counts()
    combined['Surname_Freq'] = combined['Surname'].map(surname_counts)
    
    # 3. 製品数（NumOfProducts）の非線形性を捉えるフラグ
    # クロス集計で解約率が極端だった部分を明示的にフラグ化
    combined['Is_Product_2'] = (combined['NumOfProducts'] == 2).astype(int)
    combined['Is_High_Product'] = (combined['NumOfProducts'] >= 3).astype(int)
    
    # 4. 残高（Balance）に関する特徴量
    # 残高が0かどうかのフラグ
    combined['Is_Balance_Zero'] = (combined['Balance'] == 0).astype(int)
    # 年収に対する残高の割合（分母の0割りを防ぐため +1）
    combined['Balance_to_Salary_Ratio'] = combined['Balance'] / (combined['EstimatedSalary'] + 1)
    
    # 5. 年齢と信用スコアの掛け合わせ
    combined['CreditScore_by_Age'] = combined['CreditScore'] / combined['Age']
    
    
    
    # 7. カテゴリ変数のエンコーディング（Label Encoding）
    # GBDTで扱えるように文字列を数値（0, 1, 2...）に変換
    cat_cols = ['Geography', 'Gender']
    for col in cat_cols:
        le = LabelEncoder()
        combined[col] = le.fit_transform(combined[col].astype(str))
    
    # 8. モデルの学習に不要なカラムの削除
    # 変換元のSurname、一意な値であるCustomerId、行識別用のidは削除（idは後で提出に使うため別途保持）
    drop_cols = ['id', 'CustomerId', 'Surname']
    combined = combined.drop(columns=[col for col in drop_cols if col in combined.columns])
    
    # 9. 再び train と test に分離
    X_train = combined[combined['is_train'] == 1].drop(columns=['is_train']).reset_index(drop=True)
    X_test = combined[combined['is_train'] == 0].drop(columns=['is_train']).reset_index(drop=True)
    
    print(f"前処理が完了しました。特徴量数: {X_train.shape[1]}")
    return X_train, y_train, X_test


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb  # 新しくXGBoostをインポート
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# --- 1. データの読み込み ---
train = pd.read_csv("C:\\Users\\ko.ameku.up\\Desktop\\kaggle_tutorial_zone\\data\\raw\\train.csv")
test = pd.read_csv("C:\\Users\\ko.ameku.up\\Desktop\\kaggle_tutorial_zone\\data\\raw\\test.csv")
test_ids = test['id']

# --- 2. 前処理の実行 ---
# 前回作成した前処理関数をそのまま使います（特徴量数: 17）
X_train, y_train, X_test = preprocess_data(train, test)

# --- 3. クロスバリデーションの設定（Stratified K-Fold） ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# スコア記録用とテスト予測の格納用
oof_preds_xgb = np.zeros(len(X_train))
test_preds_xgb = np.zeros(len(X_test))

# XGBoostのハイパーパラメータ設定
xgb_params = {
    'objective': 'binary:logistic',  # 二値分類
    'eval_metric': 'auc',            # 評価指標はAUC
    'learning_rate': 0.01,           # 学習率
    'random_state': 42,
    'tree_method': 'hist',           # LightGBM風の高速化アルゴリズムを使用
    'max_depth': 6                   # 木の深さの最大値（過学習防止の標準値）
}

# --- 4. 学習と予測のループ ---
for fold, (train_idx, valid_idx) in enumerate(cv.split(X_train, y_train)):
    print(f"\n--- Fold {fold + 1} の学習開始（XGBoost） ---")
    
    # データの分割
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_va, y_va = X_train.iloc[valid_idx], y_train.iloc[valid_idx]
    
    # XGBoost専用のデータセット形式（DMatrix）に変換
    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    dvalid = xgb.DMatrix(X_va, label=y_va)
    dtest = xgb.DMatrix(X_test)
    
    # モデルの訓練
    model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=1000,
        evals=[(dtrain, 'train'), (dvalid, 'valid')],
        callbacks=[xgb.callback.EarlyStopping(rounds=50, save_best=True, maximize=True)],
        verbose_eval=False  # 途中のログを非表示に
    )
    
    # 検証データでの予測結果を記録（Out-of-Fold）
    oof_preds_xgb[valid_idx] = model.predict(dvalid)
    
    # テストデータでの予測
    test_preds_xgb += model.predict(dtest) / cv.n_splits

# --- 5. ローカルスコア（CVスコア）の算出 ---
cv_score = roc_auc_score(y_train, oof_preds_xgb)
print(f"\n====================================")
print(f"XGBoost 全体の OOF ROC-AUC スコア: {cv_score:.5f}")
print(f"====================================")

# --- 6. 提出ファイルの作成 ---
submission = pd.DataFrame({
    'id': test_ids,
    'Exited': test_preds_xgb
})
submission.to_csv('submission_xgb.csv', index=False)
print("submission_xgb.csv を保存しました！")

前処理を開始します...
前処理が完了しました。特徴量数: 16

--- Fold 1 の学習開始（XGBoost） ---

--- Fold 2 の学習開始（XGBoost） ---

--- Fold 3 の学習開始（XGBoost） ---

--- Fold 4 の学習開始（XGBoost） ---

--- Fold 5 の学習開始（XGBoost） ---

XGBoost 全体の OOF ROC-AUC スコア: 0.93321
submission_xgb.csv を保存しました！
